# 07 — Qdrant Vector Memory

**Mode:** Live  
**Level:** Advanced  
**Estimated time:** 35 minutes

[Watch the original lesson video](https://www.youtube.com/watch?v=ZhEEFfIpuJg)

This notebook is meant to be read and run from top to bottom. Each code cell
changes one small part of the system, and the inspection cells show the real
Praval objects produced by that change.


## What you will build

A live persistent-memory path using OpenAI embeddings and Qdrant. You will create a unique collection, store semantic knowledge, retrieve it with a differently worded query, inspect collection metadata, test dimension compatibility, and remove the collection.


In [ ]:
from pathlib import Path
import sys

candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd() / "examples" / "notebooks",
]
support_dir = next(path for path in candidates if (path / "support.py").is_file())
if str(support_dir) not in sys.path:
    sys.path.insert(0, str(support_dir))

from support import (
    find_example_asset, get_events, require_env, setup_notebook,
    show_callout, show_flow, show_json, show_roles, show_spore,
    show_timeline, stage,
)

setup_notebook(7, 'Qdrant Vector Memory')


## Prerequisites and setup

**Course prerequisites:** Complete `course-06-agent-memory`.

**Execution requirements:** Live. Start Qdrant and set `QDRANT_URL`, `OPENAI_API_KEY`, `PRAVAL_OPENAI_MODEL`, and `PRAVAL_OPENAI_EMBEDDING_MODEL`. Optionally set `PRAVAL_OPENAI_EMBEDDING_DIMENSIONS` (default 1536).

Run the setup cell above first. It configures presentation helpers and a
credential-free lifecycle provider. It does not hide any agent workflow.


## Learning goals

- Explain how embeddings make semantic retrieval possible.
- Create and inspect a Qdrant collection.
- Use real provider embeddings for storage and search.
- Detect embedding-dimension incompatibility and clean up the collection.


## Mental model

An embedding converts text into a numeric vector. Qdrant stores that vector beside the MemoryEntry payload and compares a query vector to stored vectors. The collection's vector size and embedding identity form a compatibility contract: changing either requires a new collection or a full re-index.


In [ ]:
show_flow(
('Agent', 'semantic memory', 'agent'),
('OpenAI', 'embedding vector', 'provider'),
('Qdrant', 'persistent collection', 'memory'),
('Recall', 'similar vectors', 'agent'),
)


## Try it

We will now assemble the workflow in small steps. Run each cell, then pause at the inspection that follows it.


### Validate live configuration

Credentials are required because a local synthetic vector cannot certify the provider integration. Only non-secret model and endpoint names are displayed.


In [ ]:
import os
import uuid

values = require_env(
    "OPENAI_API_KEY", "PRAVAL_OPENAI_MODEL",
    "PRAVAL_OPENAI_EMBEDDING_MODEL", "QDRANT_URL",
)
dimensions = int(os.environ.get("PRAVAL_OPENAI_EMBEDDING_DIMENSIONS", "1536"))
collection = "praval_notebook_" + uuid.uuid4().hex[:10]
show_json(
    {
        "embedding_model": values["PRAVAL_OPENAI_EMBEDDING_MODEL"],
        "dimensions": dimensions, "collection": collection,
        "qdrant_url": values["QDRANT_URL"],
    },
    "Live vector configuration",
)


### What just happened?

The run has a unique collection name and an explicit vector dimension.

### Why this matters

Unique temporary collections prevent certification runs from contaminating one another.


### Create the Qdrant-backed Agent

Memory configuration names both the vector store and embedding runtime. Initialization creates the collection if it does not exist.


In [ ]:
from praval import Agent

learner = Agent(
    "course-qdrant-agent",
    provider="openai", model=values["PRAVAL_OPENAI_MODEL"],
    memory_enabled=True,
    memory_config={
        "backend": "qdrant", "qdrant_url": values["QDRANT_URL"],
        "collection_name": collection, "embedding_provider": "openai",
        "embedding_model": values["PRAVAL_OPENAI_EMBEDDING_MODEL"],
        "embedding_dimensions": dimensions,
    },
)
assert learner.memory is not None and learner.memory.backend == "qdrant"
long_term = learner.memory.long_term_memory
assert long_term is not None
stage("agent", "Qdrant memory ready", collection)


### What just happened?

Qdrant collection initialization and embedding runtime construction succeeded.

### Why this matters

Treat embedding model, dimensions, and collection name as one deployable configuration.


In [ ]:
show_json(learner.memory.get_memory_stats(), "Qdrant memory statistics")


### Store semantic knowledge

`remember()` sends the text to the real embedding provider, then writes the vector and typed payload to Qdrant.


In [ ]:
stored = learner.remember(
    "Agents communicate by passing structured Spores through the Reef.",
    importance=0.95,
    memory_type="semantic",
)
assert stored
stage("memory", "vector stored", stored)

after_store = learner.memory.get_memory_stats()
assert after_store["persistent_memory"]["total_memories"] >= 1
show_json(after_store, "Collection after storage")


### What just happened?

The persistent collection now contains a real vector-backed MemoryEntry.

### Why this matters

Storage evidence proves more than an API return value: it shows the remote collection accepted the entry.


### Retrieve with different wording

The query does not repeat the stored sentence. Semantic similarity should still surface the Spore-and-Reef memory.


In [ ]:
query = "How do Praval agents talk to each other?"
recalled = learner.recall(query, limit=3)
stage("memory", "semantic recall", f"{len(recalled)} matches")

assert recalled
assert any("Spores" in entry.content for entry in recalled)
show_json(
    {
        "query": query,
        "matches": [entry.content for entry in recalled],
        "backend": learner.memory.get_active_backend(),
    },
    "Live vector-memory result",
)
show_timeline()


### What just happened?

The provider embedded the query, Qdrant ranked stored vectors, and Praval returned MemoryEntry objects.

### Why this matters

Semantic retrieval lets agents recover concepts even when vocabulary changes.


### Check embedding compatibility

Opening the same collection with the wrong vector size must fail before any query or upsert can corrupt the contract.


In [ ]:
from praval.core.exceptions import EmbeddingConfigurationError
from praval.memory.long_term_memory import LongTermMemory

try:
    LongTermMemory(
        qdrant_url=values["QDRANT_URL"],
        collection_name=collection,
        embedding_provider="openai",
        embedding_model=values["PRAVAL_OPENAI_EMBEDDING_MODEL"],
        embedding_dimensions=dimensions + 1,
    )
except EmbeddingConfigurationError as error:
    compatibility_error = str(error)
else:
    raise AssertionError("Expected incompatible vector dimensions to fail")

assert "vector size" in compatibility_error
show_callout(
    "Compatibility protected",
    "Qdrant rejected a mismatched embedding dimension.",
    role="memory",
)


### What just happened?

Praval compared the existing collection configuration with the requested embedding dimensions and stopped early.

### Why this matters

Silent dimension or model changes produce invalid retrieval; compatibility checks turn them into actionable configuration errors.


## Your turn

Store two related concepts and one unrelated concept. Query for the shared idea and inspect the ranking. Keep the same embedding configuration.


In [ ]:
# learner.remember("A Reef routes Spores to handlers.", memory_type="semantic")
# learner.remember("Handlers specialize by message type.", memory_type="semantic")
# learner.remember("A bicycle has two wheels.", memory_type="semantic")
# Compare learner.recall("agent message routing", limit=3).


## Common mistake

**Mistake:** Reusing a collection after changing the embedding model or dimensions.

**Correction:** Create a versioned collection and re-index all source text when the embedding contract changes.


<details>
<summary>Under the hood</summary>

Praval stores embedding identity metadata with persistent vector data. Query embeddings must come from a compatible provider/model/dimension contract. The MemoryManager combines persistent and short-term results into one ranked list.

</details>


## Recap

- Embeddings convert text into comparable vectors.
- Qdrant persists vectors and MemoryEntry payloads.
- Collection configuration is part of the embedding contract.
- Real-service lessons must delete their temporary collections.


## Cleanup

Release every agent, transport, temporary file, or external resource created by this lesson. Cleanup is part of the example, not an afterthought.


In [ ]:
learner.memory.clear_agent_memories(learner.name)
long_term.client.delete_collection(collection_name=collection)
learner.close()
stage("agent", "closed", "Qdrant collection removed")


### Next lesson

Continue to `08_production_features.ipynb` to carry Spores across RabbitMQ, protect payloads cryptographically, and export completed spans over OTLP.
